# 📘 Databricks Learning Notes
**Module:** 02 – Get Started with Data Engineering  
**Status:** ✅ Quiz complete (10/10)  
**Last updated:** YYYY-MM-DD

**The flow in this module:**
```
Find raw data ➔ Ingest ➔ Modify (Delta) ➔ Medallion layers ➔ Declarative pipeline ➔ Automate with a Job
```
Ingestion: Data coming in
Consumption: Data going out

## 📥 Step 1: Find & Ingest Data
**Before creating a table:**
- `LIST`: shows **which files** exist in a volume
- `read_files`: previews **what's inside** the files

**Three ways to load data:**
| Method | How it works | Best for |
|---|---|---|
| **Upload UI** | Point-and-click, no code | Quick, one-off imports |
| **CTAS** (`CREATE TABLE AS SELECT`) | Creates **and** fills the table in one step | Reproducible pipelines |
| **COPY INTO** | Create an **empty table with a schema first**, then load into it | Incremental loads as new files arrive |

**COPY INTO is idempotent:** it remembers which files it already loaded. Re-running it with no new files loads **0 rows**, so there are no duplicates.

💬 **Customer framing:** "Re-run your load as often as you want. Databricks only picks up new files, so you never double-count."

## 🔺 Step 2: Delta Lake Versioning & Time Travel
- Every `INSERT`, `UPDATE`, or `DELETE` creates a **new table version**
- `DESCRIBE HISTORY` shows every version and what changed
- **Time travel** lets you query any past version
- **"Nothing is truly lost":** deleted rows still exist in earlier versions

💬 **Customer framing:** "Every change is tracked, so you can audit what happened and roll back mistakes." (Strong point for regulated industries.)


## 🥉🥈🥇 Step 3: Medallion Architecture
| Layer | Purpose | Think of it as |
|---|---|---|
| **Bronze** | Raw data, kept as-is | Safety net: reprocess anytime |
| **Silver** | Cleaned & standardized | Trusted data |
| **Gold** | Aggregated for reporting | Business-ready answers |

**Why keep Bronze?** If a cleaning rule turns out to be wrong, you can rebuild Silver and Gold from the untouched raw data.

💬 **Customer framing:** "You never lose the original data, so fixing a bad transformation doesn't mean re-ingesting from source."

## ⚙️ Step 4: Declarative Pipelines (Spark Declarative Pipelines)
You describe **what** each table should contain, and Databricks handles **how** and in what order.

| Layer | Object type | Why |
|---|---|---|
| **Bronze** | **Streaming table** | Ingests only new raw files (incremental) |
| **Silver / Gold** | **Materialized view** | Stores precomputed results that stay up to date |

**Data quality with expectations:**
- Add `CONSTRAINT ... EXPECT (...)` to a table definition
- Results show up on that table's node in the pipeline graph
- Default behavior: bad rows are **kept but tracked**
- Optional: `ON VIOLATION DROP ROW` or `ON VIOLATION FAIL UPDATE`

**Example (runs inside a pipeline, not a notebook cell):**
```sql
CREATE OR REFRESH STREAMING TABLE bronze_employees
AS SELECT * FROM STREAM read_files('/Volumes/catalog/schema/volume/raw/');

CREATE OR REFRESH MATERIALIZED VIEW silver_employees (
  CONSTRAINT valid_id EXPECT (id IS NOT NULL)
)
AS SELECT * FROM bronze_employees;
```
Bronze:

CREATE OR REFRESH creates the table on the first run and updates it on every later run. You never have to write separate "create" and "update" code.
STREAMING TABLE makes the table incremental, so it only processes new data.
bronze_employees is the table's name. The bronze_ prefix is a naming habit that shows its layer.
read_files('/Volumes/.../raw/') reads the raw files (like CSVs) sitting in that folder of your volume.
STREAM tells Databricks to remember which files it has already read. Each run picks up only new files.
SELECT * copies every column as-is, with no cleaning. That's the point of Bronze: keep the raw data untouched.

In plain English: "Watch this folder. Whenever new files show up, add their contents to bronze_employees."
Silver: 

CREATE OR REFRESH works the same way as above.
MATERIALIZED VIEW saves the query's results as stored data and keeps them up to date. Queries against it are fast because the answer is precomputed.
silver_employees is the name for the cleaned layer.
CONSTRAINT valid_id gives the quality rule a name, valid_id, which shows up in the pipeline UI.
EXPECT (id IS NOT NULL) is the rule itself: every row should have an id.
Because no action is specified, bad rows are kept but counted, so you can see how many failed.
SELECT * FROM bronze_employees reads its data from the Bronze table.

CONSTRAINT valid_id EXPECT (id IS NOT NULL) ON VIOLATION DROP ROW - this removes bada data instead of counting it

💬 **Customer framing:** "Data quality rules are built into the pipeline, so you see bad data before it reaches a dashboard."

## 🔁 Step 5: Automate with a Lakeflow Job
- A **Job** runs your pipeline steps as **tasks**
- **Task dependencies** control the order: if Silver-Gold depends on Setup-Bronze, Silver-Gold runs **only after Setup-Bronze succeeds**
- If an upstream task fails, downstream tasks don't run on bad or missing data

💬 **Customer framing:** "Orchestration is built in. No separate scheduling tool is needed."

## ⚠️ Gotchas
- **COPY INTO needs the table to exist first.** CTAS doesn't.
- **`LIST` ≠ `read_files`.** One lists file names, the other shows contents.
- **Deleting rows isn't permanent right away.** Old versions keep them until cleanup (`VACUUM`) runs.
- **Pipeline SQL (`CREATE OR REFRESH ...`) runs in a pipeline**, not as a regular notebook cell.

%sql
-- Replace the placeholders with your own catalog, schema, and volume.

-- 1. See what files exist, then preview them
-- LIST '/Volumes/workspace/learning/raw/';
-- SELECT * FROM read_files('/Volumes/workspace/learning/raw/employees.csv',
--   format => 'csv', header => true) LIMIT 10;

-- 2. CTAS: create and load in one step
-- CREATE TABLE employees AS
-- SELECT * FROM read_files('/Volumes/workspace/learning/raw/employees.csv',
--   format => 'csv', header => true);

-- 3. COPY INTO: create an empty table first, then load (safe to re-run)
-- CREATE TABLE IF NOT EXISTS employees_incremental;
-- COPY INTO employees_incremental
-- FROM '/Volumes/workspace/learning/raw/'
-- FILEFORMAT = CSV
-- FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
-- COPY_OPTIONS ('mergeSchema' = 'true');

-- 4. Version history and time travel
-- DESCRIBE HISTORY employees;
-- SELECT * FROM employees VERSION AS OF 0;
-- RESTORE TABLE employees TO VERSION AS OF 0;

%md
## 📖 Key Terms 1: The Big Picture
| Term | Definition | Simple version |
|---|---|---|
| **Lakehouse** | Combines a data lake (cheap storage for any data) with a data warehouse (fast, reliable analytics) | One platform for all data and AI |
| **Ingestion** | Bringing data **into** the platform from files, databases, apps, or streams | Data coming **in** |
| **Transformation** | Cleaning, joining, and reshaping data so it's usable | Data getting **better** |
| **Consumption** | Using prepared data in queries, dashboards, ML models, or AI apps | Data going **out** to people and tools |
| **ETL** | Extract ➔ Transform ➔ Load (clean data *before* storing it) | Traditional approach |
| **ELT** | Extract ➔ Load ➔ Transform (store raw first, clean *inside* the platform) | Lakehouse approach (this is what Bronze ➔ Silver ➔ Gold does) |

**The end-to-end flow:**
```
Sources ➔ INGEST ➔ Bronze ➔ TRANSFORM ➔ Silver ➔ Gold ➔ CONSUME
```

## 📖 Key Terms 2: Ingestion & Processing
**Processing styles**
| Term | Meaning | Example |
|---|---|---|
| **Batch** | Processes data in chunks on a schedule | Nightly sales load |
| **Streaming** | Processes data continuously as it arrives | IoT sensor feed |
| **Incremental** | Processes only what's new since the last run | COPY INTO, streaming tables |
| **Idempotent** | Running it again gives the same result, with no duplicates | Re-running COPY INTO loads 0 rows |

**Ingestion tools in Databricks**
| Tool | Best for |
|---|---|
| **Upload UI** | Quick, one-off file imports |
| **CTAS** | Creating and loading a table in one step |
| **COPY INTO** | Incremental SQL loads (thousands of files) |
| **Auto Loader** | Scalable incremental loads of files from cloud storage (millions of files) |
| **Lakeflow Connect** | Managed connectors for apps and databases (e.g., Salesforce, SQL Server) |

**Change data**
| Term | Meaning |
|---|---|
| **MERGE INTO** | Upsert: update matching rows and insert new ones in one command |
| **CDC (Change Data Capture)** | Tracking inserts, updates, and deletes from a source system |
| **AUTO CDC** (formerly `APPLY CHANGES`) | Declarative pipeline feature that applies CDC changes to a table |
| **SCD Type 1 / Type 2** | Type 1 overwrites old values. Type 2 keeps history as new rows. |

## 📖 Key Terms 3: Delta Lake & Storage
**Delta Lake features**
| Term | Meaning |
|---|---|
| **Delta Lake** | The default table format in Databricks: open files plus a transaction log |
| **Transaction log** | Records every change to a table; powers versioning and time travel |
| **ACID transactions** | Writes fully succeed or fully fail, so readers never see half-written data |
| **Schema enforcement** | Blocks writes that don't match the table's columns and types |
| **Schema evolution** | Allows new columns to be added safely (e.g., `mergeSchema`) |
| **Time travel** | Query or restore past versions (`VERSION AS OF`, `RESTORE`) |
| **VACUUM** | Permanently deletes old files no longer needed; limits how far back time travel works |
| **OPTIMIZE** | Compacts many small files into fewer large ones for faster queries |
| **Liquid clustering** | Organizes data automatically for faster filtering; replaces manual partitioning |

**Tables and storage**
| Term | Meaning |
|---|---|
| **Managed table** | Databricks controls the data files. Dropping the table deletes the data. |
| **External table** | Data lives in your own storage location. Dropping the table keeps the files. |
| **Volume** | Unity Catalog location for non-table files (CSV, JSON, images, PDFs) |
| **View** | Saved query. Runs fresh each time and stores no data. |
| **Materialized view** | Saved query whose results are stored and kept up to date |
| **Streaming table** | Table that ingests new data incrementally |

## 📖 Key Terms 4: Pipelines, Orchestration & Compute
**Lakeflow (Databricks' data engineering suite)**
| Component | Role |
|---|---|
| **Lakeflow Connect** | Ingest: brings data in |
| **Spark Declarative Pipelines** (formerly DLT) | Transform: you define *what* tables should contain, Databricks handles *how* |
| **Lakeflow Jobs** | Orchestrate: schedule and run tasks in order |

**Pipeline concepts**
| Term | Meaning |
|---|---|
| **Declarative** | Describe the result you want, not the step-by-step process |
| **Expectations** | Data quality rules (`CONSTRAINT ... EXPECT`): keep and track, drop the row, or fail the update |
| **Pipeline graph (DAG)** | Visual map of how tables depend on each other |
| **Task** | One step in a Job (a notebook, pipeline, SQL query, etc.) |
| **Task dependency** | A task runs only after its upstream task succeeds |
| **Trigger / schedule** | What starts a Job: a time schedule, a file arriving, or a manual run |

**Compute**
| Term | Meaning |
|---|---|
| **Serverless compute** | Databricks manages the infrastructure; starts fast, no setup |
| **SQL warehouse** | Compute built for SQL queries and dashboards |
| **Photon** | Databricks' high-performance query engine that speeds up SQL and DataFrame workloads |

**Governance**
| Term | Meaning |
|---|---|
| **Unity Catalog** | One place to govern, secure, and discover all data and AI assets |
| **Lineage** | Automatically tracks where data came from and where it's used |

💬 **Customer framing:** "Lakeflow covers ingest, transform, and orchestrate in one platform, and Unity Catalog governs all of it, including lineage for audits."

note: Some names have changed recently. Delta Live Tables (DLT) became Lakeflow Declarative Pipelines and then Spark Declarative Pipelines, and APPLY CHANGES became AUTO CDC. Customers may use the older names, so it's worth knowing both.